In [5]:
# Required Python packages
from datetime import date, datetime, timedelta
import calendar
import pandas as pd
import numpy as np
import itertools
import copy
from numpy.linalg import multi_dot
from scipy.stats import norm
from scipy.stats import bernoulli
from scipy.optimize import fmin_slsqp as min


from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, r2_score
import statsmodels.api as sm
from statsmodels.api import OLS

from sklearn.linear_model import LassoCV
from yellowbrick.datasets import load_concrete
from yellowbrick.regressor import AlphaSelection

import matplotlib.pyplot as plt
import math


#Import Date opertative user defined functions
from ipynb.fs.full.user_defined_vik_functions import get_all_monthly_option_expiries, \
                                                     find_last_thurs_date_of_month, \
                                                     prev_workday_if_holiday, find_wkly_expries,\
                                                     date_of_prev_thurs

# Import dataframe naming functions 
from ipynb.fs.full.user_defined_vik_functions import get_mthly_df_name_from_expiry

#Import data loading functions
from ipynb.fs.full.user_defined_vik_functions import load_all_mthly_data

#import traded options parameters and info
from ipynb.fs.full.user_defined_vik_functions import generate_weekly_strikes, generate_mthly_strikes

#Import pricing functions
from ipynb.fs.full.user_defined_vik_functions import BSM_call_vec_with_div, BSM_put_vec_with_div

#Import implied volatility processing functions
from ipynb.fs.full.user_defined_vik_functions import generate_weekly_iv, gen_interpolated_iv

#Import Risk-free interest rate generation functions
from ipynb.fs.full.user_defined_vik_functions import generate_weekly_ir

#Import Stochatic process simulation functions
from ipynb.fs.full.user_defined_vik_functions import gbm1W_simulation, generate_covariance_from_correlation

# Import risk free interest rate function
from ipynb.fs.full.user_defined_vik_functions import get_risk_free_rate_from_exact_date


In [6]:
# Stock Index of Interest
stock_ident = "BANKNIFTY"

#Static hedging performed at different moneyness regions 
#i.e. moneyness is used to select the option with nearest moneyness match
# ATM - At the Money, ITM - In the money, OTM - Out of the Money
prod_moneyness = "ITM"

#Product type to hedge: either "CE" or "PE"
# prod_type = "PE"
prod_type = "PE"

# The scope of this code is to hedge one option 
# and scope will be extended to a portfolio
no_of_assets = 1
cor_mat = [[1]]

#Path to refer data
source_path = "/home/jupyter-partha/Vikranth - Chapter 2/"
input_sub_path = "Input Data/mkt_data_covid_region/"
output_sub_path = "Output Data/"
input_data_path = source_path + input_sub_path
output_data_path = source_path + output_sub_path


# Periods of interest will be a dictionary
#Key is the year, value is a list of months 1-12, 1- Jan, 2 - Feb,...12 - Dec
# For E.g., periods_of_interest = {2020: [3], 2019: [11, 12]}
periods_of_interest = {2019: [8, 9, 10, 11, 12], 2020: [1, 2, 3, 4, 5, 6, 7]}

#List of holidays
holidays_list = [date(2019, 3, 4), date(2019, 3, 21),\
                 date(2019, 4, 17), date(2019, 4, 19), date(2019, 4, 29),\
                 date(2019, 5, 1),\
                 date(2019, 6, 5),\
                 date(2019, 8, 12), date(2019, 8, 15),\
                 date(2019, 9, 2), date(2019, 9, 10), \
                 date(2019, 10, 2), date(2019, 10, 8), date(2019, 10, 21), date(2019, 10, 28), \
                 date(2019, 11, 12), \
                 date(2019, 12, 25), \
                 date(2020, 2, 21), \
                 date(2020, 3, 10), \
                 date(2020, 4, 2), date(2020, 4, 6), date(2020, 4, 10), date(2020,4, 14), \
                 date(2020, 5, 1), date(2020, 5, 25), \
                 date(2020, 10, 2), date(2020, 11, 16), date(2020, 11, 30), date(2020, 12, 25)]
#Simulation Parameters
no_of_paths = 5000
no_opt=1


In [7]:
#########################
#User-defined functions #
#########################

def generate_weekly_static_pfl_weights(no_of_paths, mthly_expiries_list, \
                                       dict_wkly_expiries_each_month, \
                                       dict_ce_wkly_strikes, dict_pe_wkly_strikes, \
                                       dict_wkly_spots, dict_wkly_iv, \
                                       prod_type, prod_moneyness, output_path, \
                                       option_type = "CE", no_of_assets=1, cor_mat=[[1]],\
                                       stock_ident="BANKNIFTY"):
    ce_pfl_weights = []
    pe_pfl_weights = [] 
    cash_weights = []
    for each_month in mthly_expiries_list:
        strike = dict_mthly_strikes[each_month.strftime("%d-%b-%Y")]
        weekly_pfl_weights = []
        ce_dict = {}
        pe_dict = {}
        cash_dict = {}
        week_list = dict_wkly_expiries_each_month[each_month.strftime("%d-%b-%Y")]
        no_of_weeks = len(week_list)
        
        for week in range(0, no_of_weeks):
            s_t = dict_wkly_spots[each_month.strftime("%d-%b-%Y")][week]
            K = strike
            vol_list = np.array([dict_wkly_iv[each_month.strftime("%d-%b-%Y")][week]])
            r = np.array([dict_wkly_ir[each_month.strftime("%d-%b-%Y")][week]])
            
            if (week == no_of_weeks-1):
                sim_start_date = dict_wkly_expiries_each_month[each_month.strftime("%d-%b-%Y")][week]
                sim_end_date = each_month
            else:
                sim_start_date = dict_wkly_expiries_each_month[each_month.strftime("%d-%b-%Y")][week]
                sim_end_date = dict_wkly_expiries_each_month[each_month.strftime("%d-%b-%Y")][week + 1]
         
            sim_stock_mat = gbm1W_simulation(no_of_paths, sim_start_date, sim_end_date, \
                                             no_of_assets, s_t, r, vol_list, cor_mat)
            
            call_strikes = np.sort(np.array(dict_ce_wkly_strikes[each_month.strftime("%d-%b-%Y")][week])).reshape(1,-1)
            put_strikes = np.sort(np.array(dict_pe_wkly_strikes[each_month.strftime("%d-%b-%Y")][week])).reshape(1,-1)
            no_of_calls = call_strikes.size
            no_of_puts = put_strikes.size
            no_ind_vars = no_of_calls + no_of_puts
            
            
            stock_vec = sim_stock_mat[:, 1]
            r_f = get_risk_free_rate_from_exact_date(sim_start_date)
            dt = float((each_month - sim_end_date).days) / 365
            opt_strike = strike
            dt = float((each_month - sim_end_date).days) / 365

            df_daily_iv_data = pd.read_csv(output_path + "A1_" + stock_ident + "_" +  "Daily_vol_smile_" + option_type + "_" + each_month.strftime("%d-%b-%Y") + ".csv")
            df_daily_iv_data = df_daily_iv_data[df_daily_iv_data['Impl_Vol'] > 0.00001]
            
            spot_list = sim_stock_mat[:,1]
            if (prod_type == "CE"):
                mny_list = spot_list / K
            else:
                mny_list = np.reciprocal(spot_list) * K

            if (week != no_of_weeks-1):
                date_str = sim_start_date.strftime("%d-%b-%Y")
                gen_iv = lambda x: gen_interpolated_iv(df_daily_iv_data, date_str, x)
                gen_iv = np.vectorize(gen_iv)
                vol_arr = gen_iv(mny_list)
            else:
                vol_arr = np.array([0]) 
       
            if (option_type == "CE" and week != no_of_weeks-1):
                option_value = BSM_call_vec_with_div(stock_vec, opt_strike, r, r_f, vol_arr, dt)
            elif (option_type == "PE" and week != no_of_weeks-1):
                option_value = BSM_put_vec_with_div(stock_vec, opt_strike, r, r_f, vol_arr, dt) 
            elif (option_type == "CE" and week == no_of_weeks-1):
                option_value = np.maximum(stock_vec - opt_strike, 0) 
            else:
                option_value = np.maximum(opt_strike - stock_vec, 0)
    
            option_value = np.asarray(option_value)
            option_value = option_value.reshape(-1, 1) 
            
            y = option_value
            stock_vec = stock_vec.reshape(-1,1)
            x = np.concatenate((np.maximum(stock_vec-call_strikes,0), np.maximum(put_strikes-stock_vec,0)), axis=1) 

            print("SKLearn Linear regression - Lasso")
            lin_model_lasso = Lasso(alpha=1.0).fit(x, y)
            pfl_weights = lin_model_lasso.coef_
            pfl_weights = pfl_weights.reshape(-1)
            cash_weight = lin_model_lasso.intercept_
            print("R-Square: ", lin_model_lasso.score(x, y))
            print("Coefficients:")
            print(pfl_weights)
            print(cash_weight)

#             #Find Optimal Alpha
#             alphas = np.array([-1.0, -0.5, 0.0, 0.25, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0])
#             model = LassoCV(alphas=alphas)
#             visualizer = AlphaSelection(model)
#             visualizer.fit(x, y)
#             visualizer.show()    

            # Call Pfl weights
            call_strikes = call_strikes.reshape(-1) 
            ce_dates_list = np.array([sim_start_date.strftime("%d-%b-%Y") for i in range(0, no_of_calls)]).reshape(-1)
            ce_wt_list = pfl_weights[0:no_of_calls]
            ce_dict = {"Date": ce_dates_list, "Strike": call_strikes, "Pfl_weights": ce_wt_list}
            df_ce = pd.DataFrame(ce_dict)
            ce_pfl_weights.append(df_ce)
    
            
            #Put Pfl Weights
            put_strikes = put_strikes.reshape(-1) 
            pe_dates_list = np.array([sim_start_date.strftime("%d-%b-%Y") for i in range(0, no_of_puts)]).reshape(-1)
            pe_wt_list = pfl_weights[no_of_calls:no_ind_vars]
            pe_dict = {"Date": pe_dates_list, "Strike":put_strikes, "Pfl_weights": pe_wt_list}
            df_pe = pd.DataFrame(pe_dict)
            pe_pfl_weights.append(df_pe)
            
            # Cash component
            cash_date = [sim_start_date.strftime("%d-%b-%Y")]
            cash_dict = {"Date":cash_date, "Pfl_weights":cash_weight} 
            df_cash = pd.DataFrame(cash_dict)
            cash_weights.append(df_cash)

            
        df_ce_pfl_weights = pd.concat(ce_pfl_weights, axis=0)
        df_pe_pfl_weights = pd.concat(pe_pfl_weights, axis=0)
        df_cash_pfl_weights = pd.concat(cash_weights, axis=0)
    
        df_ce_pfl_weights.to_csv(output_path  + "B3_LR_" + stock_ident + "_" + prod_moneyness + "_" + prod_type + "_CE_Pfl_weights_const_Vol_Smile.csv", index = False)
        df_pe_pfl_weights.to_csv(output_path + "B3_LR_" + stock_ident + "_" + prod_moneyness + "_" + prod_type + "_PE_Pfl_weights_const_Vol_Smile.csv", index = False)
        df_cash_pfl_weights.to_csv(output_path  + "B3_LR_" + stock_ident + "_" + prod_moneyness + "_" + prod_type + "_Cash_Pfl_weights_const_Vol_Smile.csv", index = False)
    
    return df_ce_pfl_weights, df_pe_pfl_weights, df_cash_pfl_weights

def output_strikes_hedged(prod_type, prod_moneyness, dict_mthly_strikes, stock_ident, output_path):
    list_of_dates = []
    list_of_prod_type = []
    list_of_strikes = []
    for each_month in dict_mthly_strikes.keys():
        list_of_dates.append(each_month)
        list_of_strikes.append(dict_mthly_strikes[each_month])
        list_of_prod_type.append(prod_type)
    dict_hedged_prod = {"Date":list_of_dates, "prod_type":list_of_prod_type, "Strike":list_of_strikes}
    df_hedged_prod = pd.DataFrame(dict_hedged_prod) 
    df_hedged_prod.to_csv(output_path + "B3_LR_" + stock_ident + "_" + prod_moneyness + "_" + prod_type + "_strikes_hedged.csv", index = False)
    return(df_hedged_prod)



In [8]:

#Find the monthly strikes of option from mkt data to find the option to be hedged
#Every month, an option is hedged
mthly_expiries_list = get_all_monthly_option_expiries(periods_of_interest, holidays_list)
dict_wkly_expiries_each_month = find_wkly_expries(mthly_expiries_list, holidays_list)

#Load all monthly mkt data
mthly_mkt_data = load_all_mthly_data(mthly_expiries_list, input_data_path, holidays_list, prod_type_lists=["FUT", "CE", "PE"], stock_ident=stock_ident)

#Load weekly strikes of short term options used for hedging monthly options 
dict_ce_wkly_strikes, dict_pe_wkly_strikes, dict_wkly_spots = generate_weekly_strikes(dict_wkly_expiries_each_month, input_data_path, stock_ident = "BANKNIFTY")     

#Load monthly strikes of interest as of required moneyness on the begining of month
dict_mthly_strikes = generate_mthly_strikes(mthly_mkt_data, prod_moneyness, prod_type, holidays_list, stock_ident=stock_ident)

#Load weekly IV for Black-Scholes model input and monte-carlo simulation of stock paths
dict_wkly_iv = generate_weekly_iv(dict_wkly_expiries_each_month, dict_wkly_spots, dict_mthly_strikes, prod_type, output_path=output_data_path, atm_ind=0, stock_ident=stock_ident)

#Load weekly risk free interest rates from futures
#This function to be changed based on updates on risk-free-interest - now kept it as cosntant
dict_wkly_ir = generate_weekly_ir(dict_wkly_expiries_each_month, mthly_mkt_data, stock_ident)

#Load Static portfolio weights built by neural network
ce_pfl_wts, pe_pfl_wts, cash_pfl_wts = generate_weekly_static_pfl_weights(no_of_paths, mthly_expiries_list, dict_wkly_expiries_each_month, \
                                                        dict_ce_wkly_strikes, dict_pe_wkly_strikes, dict_wkly_spots, \
                                                        dict_wkly_iv, prod_type, prod_moneyness, output_data_path, \
                                                        option_type = prod_type, no_of_assets=no_of_assets, \
                                                        cor_mat=cor_mat, stock_ident=stock_ident) 
# The strike correspnding to the option that is hedged
df_hedged_prod = output_strikes_hedged(prod_type, prod_moneyness, dict_mthly_strikes, stock_ident, output_data_path)


SKLearn Linear regression - Lasso
R-Square:  0.9999470659510644
Coefficients:
[-9.99395399e-01  1.77456715e-01 -5.46894676e-02 -1.71543249e-01
  6.44172629e-02  4.25472530e-01 -2.11850425e-02 -8.66272661e-02
  8.36414658e-02 -2.89960652e-02 -0.00000000e+00  1.18524960e-01
 -0.00000000e+00 -0.00000000e+00  0.00000000e+00  3.13182055e-02
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  3.25879893e-04  1.18125523e-02
  3.01178758e-03  0.00000000e+00  2.96268084e-02  9.01674277e-03
 -1.94019196e-01  1.04392233e+00  2.31343699e-01 -1.48272406e-01]
[1488.8568284]
SKLearn Linear regression - Lasso
R-Square:  0.9995220636263802
Coefficients:
[-1.22977947 -0.00791853 -0.35417675 -0.16938304  0.40370956  0.8266216
  0.72914888 -0.18615187 -1.31638488 -0.7

/opt/tljh/user/lib/python3.7/site-packages/sklearn/linear_model/_coordinate_descent.py:532: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 1021063.298793807, tolerance: 404529.46140971774
  positive)


SKLearn Linear regression - Lasso
R-Square:  0.9999464766403572
Coefficients:
[-1.13896809  0.11487772  0.00128445 -0.02877872 -0.02456669  0.07186456
  0.21359823  0.14885554 -0.20811654 -0.33518397  0.29248014  0.25917028
 -0.25369616 -0.11828545  0.02605672  0.01359257  0.09296192  0.31112566
 -0.14730631  0.20187836  0.          0.          0.          0.
  0.          0.          0.          0.          0.00115815  0.
  0.         -0.02708977  0.02239075  0.05468691 -0.04114811 -0.07892983
  0.02686995  0.12146918  0.0309158  -0.14031435 -0.05823928  0.40041778
  0.62967804  0.10361723 -0.0451722 ]
[2433.31043553]
SKLearn Linear regression - Lasso
R-Square:  0.9999597957964014
Coefficients:
[-1.13568588e+00  1.29826925e-01  1.35162317e-02 -1.98094641e-03
 -2.73410301e-03  0.00000000e+00  4.54980820e-04  5.20764996e-04
  0.00000000e+00 -0.00000000e+00 -7.92267827e-04 -0.00000000e+00
  3.96474882e-03  5.23793576e-02  4.30075755e-02  0.00000000e+00
  0.00000000e+00  7.34623399e-03 -0

/opt/tljh/user/lib/python3.7/site-packages/sklearn/linear_model/_coordinate_descent.py:532: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 229369.7721244546, tolerance: 135336.80729846252
  positive)


SKLearn Linear regression - Lasso
R-Square:  0.999971813254224
Coefficients:
[-1.00599269  0.09576386  0.01188054  0.0279062   0.03641144  0.01104557
 -0.06919437  0.03324737  0.17863192  0.06449754 -0.01239298 -0.07754441
  0.09356485  0.11092507 -0.          0.          0.03687568  0.04485391
  0.          0.09333582  0.0878005   0.          0.          0.
  0.          0.          0.          0.         -0.01503783  0.03743564
  0.01711443 -0.05899688 -0.01787759  0.0959052   0.04283049 -0.12694123
 -0.05920586  0.42123193  0.61090442  0.08066183 -0.02844372]
[1683.31949032]
SKLearn Linear regression - Lasso
R-Square:  0.9999297078579665
Coefficients:
[-8.34242594e-01  4.59295834e-02 -3.34429762e-02  8.11770799e-02
  6.82796449e-02  2.89065089e-02  7.68179064e-02  1.02192647e-01
  3.95306666e-02  1.15929368e-02  8.16760110e-02  8.30307199e-02
  3.29932799e-02  1.27396056e-02  3.34188275e-02  5.53450919e-02
  6.99199865e-02  6.05695995e-03  0.00000000e+00  0.00000000e+00
  0.00000000

...

SKLearn Linear regression - Lasso


/opt/tljh/user/lib/python3.7/site-packages/sklearn/linear_model/_coordinate_descent.py:532: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 216057.5780720053, tolerance: 148320.13692844132
  positive)


R-Square:  0.9997272148598895
Coefficients:
[-0.43098563  0.06526902  0.03655584  0.01852087  0.0056161   0.00564571
  0.01463619  0.01730022  0.03581491  0.03600323  0.020718    0.00159599
 -0.02354215 -0.02497536  0.01121003  0.07682311  0.10817103  0.11763613
  0.00047275 -0.34419642 -0.2665033   0.39477959  0.10507271 -0.03288398
  0.03675976 -0.00671565  0.00105608  0.          0.          0.
  0.00090564 -0.05757294  0.30385834  0.08584259 -0.01188694 -0.13839833
 -0.15738213  0.11070362  0.33480297 -0.06625693 -0.14724437 -0.00418374
  0.07980337  0.11917069  0.06300046 -0.01799365 -0.04105564 -0.04002662
  0.00979904  0.13056711  0.19369079  0.15638689  0.06574805]
[584.28512676]
SKLearn Linear regression - Lasso
R-Square:  0.9998831370126035
Coefficients:
[-3.89443302e-01  6.16755628e-02  3.15151942e-02  2.88302654e-02
  3.00251903e-02  2.16925246e-02  1.76790565e-02  1.20515704e-02
  4.70159085e-03  9.14199016e-03  1.82242902e-02  2.68342653e-02
  3.41528030e-02  2.47264855e-

/opt/tljh/user/lib/python3.7/site-packages/sklearn/linear_model/_coordinate_descent.py:532: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 1111092.8710149298, tolerance: 281639.89575437916
  positive)


R-Square:  0.999217094219299
Coefficients:
[-4.60602751e-01  1.63980932e-01  7.45590700e-02 -1.85542855e-02
 -1.01610173e-01 -1.24068950e-01 -1.82427738e-01 -4.12314728e-01
  2.98470181e-01  2.71505138e-01  6.75369573e-02  1.78931332e-02
 -6.33477072e-02 -5.57661749e-02  4.60106047e-02  1.35835443e-01
  3.09426696e-02 -3.16895844e-02 -1.28505033e-02  1.19777267e-02
  3.90444848e-02 -0.00000000e+00 -4.60453144e-03 -1.97808035e-02
 -0.00000000e+00  5.85370622e-02  3.94742461e-02  0.00000000e+00
  0.00000000e+00  0.00000000e+00  8.25900248e-03  1.31741650e-02
  1.78303545e-02  1.13001323e-02  0.00000000e+00  6.82203439e-02
  0.00000000e+00  0.00000000e+00  0.00000000e+00  5.68305631e-03
  3.55426219e-02  8.72176909e-03 -9.33073322e-02  3.92658173e-02
  1.26171762e-01  9.36655888e-02 -9.00954521e-02 -1.00746107e-01
 -1.77932543e-02  8.78556308e-02  1.42021205e-01  8.21228356e-02
 -6.08817833e-02 -1.52923030e-01 -1.21369095e-01  1.04442731e-02
  1.69192110e-01  2.31559432e-01  1.44887055e-0